# Potter Airlines Pricing Model

In [117]:
import numpy as np
import pandas as pd


In [118]:
flights = pd.read_csv("flights.csv")
flights.head()


,flight_id,origin,destination,departure_date,base_fare,seats_remaining,capacity,route_demand,season,is_weekend
0,PA101,Quebec City,Edmonton,2027-02-16,370.52,86,148,0.55,Regular,False
1,PA102,Ottawa,Winnipeg,2027-12-10,433.00,20,247,0.38,Vacation,True
2,PA103,Vancouver,Winnipeg,2027-01-28,244.08,8,216,0.26,Vacation,False
3,PA104,Calgary,Vancouver,2027-01-03,169.16,81,116,0.24,Vacation,True
4,PA105,Vancouver,Calgary,2027-07-15,790.30,177,291,0.31,Vacation,False


In [119]:
flights.info()
flights.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   flight_id        200 non-null    object 
 1   origin           200 non-null    object 
 2   destination      200 non-null    object 
 3   departure_date   200 non-null    object 
 4   base_fare        200 non-null    float64
 5   seats_remaining  200 non-null    int64  
 6   capacity         200 non-null    int64  
 7   route_demand     200 non-null    float64
 8   season           200 non-null    object 
 9   is_weekend       200 non-null    bool   
dtypes: bool(1), float64(2), int64(2), object(5)
memory usage: 14.4+ KB


Index(['flight_id', 'origin', 'destination', 'departure_date', 'base_fare',
       'seats_remaining', 'capacity', 'route_demand', 'season', 'is_weekend'],
      dtype='object')

## Time Factor


The time factor adjusts the base fare based on how close the flight is to departure.
Flights closer to departure receive a higher multiplier, while flights booked more
than 30 days in advance receive a small discount.

In [120]:
# calculate days until departure first
flights["departure_date"] = pd.to_datetime(flights["departure_date"])

pricing_date = pd.Timestamp("2027-01-1")

flights["days_until_departure"] = (
    flights["departure_date"] - pricing_date
).dt.days

flights[["flight_id", "departure_date", "days_until_departure"]].head()

,flight_id,departure_date,days_until_departure
0,PA101,2027-02-16,46
1,PA102,2027-12-10,343
2,PA103,2027-01-28,27
3,PA104,2027-01-03,2
4,PA105,2027-07-15,195


In [121]:
assert flights.notna().all().all()
assert flights["flight_id"].is_unique
assert (flights["capacity"] > 0).all()
assert flights["seats_remaining"].between(0, flights["capacity"]).all()
assert (flights["days_until_departure"] >= 0).all()
assert (flights["base_fare"] > 0).all()
assert flights["route_demand"].between(0, 1).all()

print("Input checks passed.")

Input checks passed.


In [122]:
time_factor_table = pd.DataFrame({
    "Days Until Departure": [
        "Same day", "1-7 days", "8-14 days", "15-30 days", "31+ days"
    ],
    "Multiplier": [
        1.45, 1.25, 1.18, 1.08, 0.95
    ]
})

time_factor_table

,Days Until Departure,Multiplier
0,Same day,1.45
1,1-7 days,1.25
2,8-14 days,1.18
3,15-30 days,1.08
4,31+ days,0.95


In [123]:
days = flights["days_until_departure"]


flights["time_factor"] = np.select(
    [
        days == 0,
        days <= 7,
        days <= 14,
        days <= 30
    ],
    [
        1.45,
        1.25,
        1.18,
        1.08
    ],
    default=0.95
)

flights[
    ["flight_id", "days_until_departure", "time_factor"]
].head(10)



,flight_id,days_until_departure,time_factor
0,PA101,46,0.95
1,PA102,343,0.95
2,PA103,27,1.08
3,PA104,2,1.25
4,PA105,195,0.95
5,PA106,255,0.95
6,PA107,56,0.95
7,PA108,312,0.95
8,PA109,136,0.95
9,PA110,310,0.95


## Demand Factor

The demand factor adjusts the fare based on the level of demand for a route.
Low-demand routes receive a discount to encourage bookings, while high-demand
routes receive a higher multiplier.

In [124]:
demand_factor_table = pd.DataFrame({
    "Route Demand": [
        "Below 0.40",
        "0.40-0.70",
        "Above 0.70"
    ],
    "Demand Level": [
        "Low",
        "Normal",
        "High"
    ],
    "Multiplier": [
        0.90,
        1.00,
        1.20
    ]
})

demand_factor_table

,Route Demand,Demand Level,Multiplier
0,Below 0.40,Low,0.9
1,0.40-0.70,Normal,1.0
2,Above 0.70,High,1.2


Low-demand routes receive a 10% discount to encourage bookings. Normal-demand routes maintain the base fare, while high-demand routes receive a 20% increase to reflect stronger customer demand.

In [125]:
demand = flights["route_demand"]

flights["demand_factor"] = np.select(
    [
        demand < 0.40,
        demand <= 0.70
    ],
    [
        0.90,
        1.00
    ],
    default=1.20
)

flights[
    ["flight_id", "route_demand", "demand_factor"]
].head(10)

flights["demand_factor"].value_counts()

demand_factor
0.9    86
1.2    59
1.0    55
Name: count, dtype: int64

## Capacity Factor

The capacity factor adjusts the fare based on how full a flight is. Flights with lower occupancy receive a discount to encourage bookings. As the flight fills up and seats become more limited, the fare increases.

In [126]:
flights["load_factor"] = (
    1 - flights["seats_remaining"] / flights["capacity"]
)

In [127]:
capacity_factor_table = pd.DataFrame({
    "Load Factor": [
        "Below 50%",
        "50%-79%",
        "80%-89%",
        "90%+"
    ],
    "Occupancy Level": [
        "Low",
        "Normal",
        "High",
        "Very High"
    ],
    "Multiplier": [
        0.90,
        1.00,
        1.15,
        1.30
    ]
})

capacity_factor_table

,Load Factor,Occupancy Level,Multiplier
0,Below 50%,Low,0.90
1,50%-79%,Normal,1.00
2,80%-89%,High,1.15
3,90%+,Very High,1.30


In [128]:
load = flights["load_factor"]

flights["capacity_factor"] = np.select(
    [
        load < 0.50,
        load < 0.80,
        load < 0.90
    ],
    [
        0.90,
        1.00,
        1.15
    ],
    default=1.30
)

flights[
    [
        "flight_id",
        "capacity",
        "seats_remaining",
        "load_factor",
        "capacity_factor"
    ]
].head(10).round(2)

,flight_id,capacity,seats_remaining,load_factor,capacity_factor
0,PA101,148,86,0.42,0.9
1,PA102,247,20,0.92,1.3
2,PA103,216,8,0.96,1.3
3,PA104,116,81,0.30,0.9
4,PA105,291,177,0.39,0.9
5,PA106,172,47,0.73,1.0
6,PA107,106,41,0.61,1.0
7,PA108,121,2,0.98,1.3
8,PA109,266,142,0.47,0.9
9,PA110,146,140,0.04,0.9


## Seasonal Factor

The seasonal factor adjusts the fare based on the travel season.
Regular-season flights keep the standard fare, while flights during vacation
periods receive a higher multiplier due to increased travel demand.

In [129]:
seasonal_factor_table = pd.DataFrame({
    "Season": [
        "Regular",
        "Vacation"
    ],
    "Multiplier": [
        1.00,
        1.15
    ]
})

seasonal_factor_table

,Season,Multiplier
0,Regular,1.00
1,Vacation,1.15


In [130]:
flights["seasonal_factor"] = np.where(
    flights["season"] == "Vacation",
    1.15,
    1.00
)

flights[
    ["flight_id", "season", "seasonal_factor"]
].head(10)



,flight_id,season,seasonal_factor
0,PA101,Regular,1.00
1,PA102,Vacation,1.15
2,PA103,Vacation,1.15
3,PA104,Vacation,1.15
4,PA105,Vacation,1.15
5,PA106,Regular,1.00
6,PA107,Regular,1.00
7,PA108,Regular,1.00
8,PA109,Regular,1.00
9,PA110,Regular,1.00


## Weekend Factor
Weekend flights receive a 10% fare increase to reflect higher travel demand, while weekday fares remain unchanged.

In [131]:
flights["weekend_factor"] = np.where(
    flights["is_weekend"],
    1.10,
    1.00
)

flights["is_weekend"].value_counts()

is_weekend
False    118
True      82
Name: count, dtype: int64

In [132]:
weekend_factor_table = pd.DataFrame({
    "Is Weekend": [False, True],
    "Multiplier": [1.00, 1.10]
})

weekend_factor_table

,Is Weekend,Multiplier
0,False,1.0
1,True,1.1


In [133]:
flights["weekend_factor"] = np.where(
    flights["is_weekend"] == 1,
    1.10,
    1.00
)

flights[
    ["flight_id", "is_weekend", "weekend_factor"]
].head(10)

,flight_id,is_weekend,weekend_factor
0,PA101,False,1.0
1,PA102,True,1.1
2,PA103,False,1.0
3,PA104,True,1.1
4,PA105,False,1.0
5,PA106,False,1.0
6,PA107,True,1.1
7,PA108,False,1.0
8,PA109,False,1.0
9,PA110,True,1.1


## Combine Factors

In [134]:
flights["adjusted_fare"] = (
    flights["base_fare"]
    * flights["time_factor"]
    * flights["demand_factor"]
    * flights["capacity_factor"]
    * flights["seasonal_factor"]
    * flights["weekend_factor"]
)

flights["adjusted_fare"] = flights["adjusted_fare"].round(2)


flights[
    [
        "flight_id",
        "base_fare",
        "time_factor",
        "demand_factor",
        "capacity_factor",
        "seasonal_factor",
        "weekend_factor",
        "adjusted_fare"
    ]
].head(10)

,flight_id,base_fare,time_factor,demand_factor,capacity_factor,seasonal_factor,weekend_factor,adjusted_fare
0,PA101,370.52,0.95,1.0,0.9,1.00,1.0,316.79
1,PA102,433.00,0.95,0.9,1.3,1.15,1.1,608.82
2,PA103,244.08,1.08,0.9,1.3,1.15,1.0,354.68
3,PA104,169.16,1.25,0.9,0.9,1.15,1.1,216.66
4,PA105,790.30,0.95,0.9,0.9,1.15,1.0,699.36
5,PA106,647.39,0.95,1.2,1.0,1.00,1.0,738.02
6,PA107,427.78,0.95,1.0,1.0,1.00,1.1,447.03
7,PA108,221.55,0.95,1.0,1.3,1.00,1.0,273.61
8,PA109,479.07,0.95,1.0,0.9,1.00,1.0,409.60
9,PA110,780.55,0.95,1.0,0.9,1.00,1.1,734.11


## Set Maximum and Mininum Bounds

To prevent unrealistic ticket prices, the final fare is bounded between
$250 and $600. Any adjusted fare below $250 is increased to $250, while
any adjusted fare above $600 is capped at $600.

In [135]:
print("Minimum base fare:", flights["base_fare"].min())
print("Maximum base fare:", flights["base_fare"].max())

MIN_FARE = 200
MAX_FARE = 600

Minimum base fare: 102.07
Maximum base fare: 799.71


In [136]:
flights["final_fare"] = np.clip(
    flights["adjusted_fare"],
    MIN_FARE,
    MAX_FARE
).round(2)

assert flights["final_fare"].between(MIN_FARE, MAX_FARE).all()

print("Final fare checks passed.")

Final fare checks passed.


In [137]:
print(
    "Minimum bound:",
    (flights["adjusted_fare"] < MIN_FARE).sum()
)

print(
    "Maximum bound:",
    (flights["adjusted_fare"] > MAX_FARE).sum()
)


Minimum bound: 24
Maximum bound: 69


In [138]:
flights[
    flights["adjusted_fare"] != flights["final_fare"]
][
    ["flight_id", "base_fare", "adjusted_fare", "final_fare"]
].head(10)

,flight_id,base_fare,adjusted_fare,final_fare
1,PA102,433.00,608.82,600.0
4,PA105,790.30,699.36,600.0
5,PA106,647.39,738.02,600.0
9,PA110,780.55,734.11,600.0
10,PA111,747.61,970.32,600.0
13,PA114,727.87,622.33,600.0
14,PA115,113.08,110.07,200.0
16,PA117,659.39,641.86,600.0
17,PA118,113.95,134.45,200.0
20,PA121,794.11,611.07,600.0


## Final Pricing Table

The table below summarizes the original base fare, all dynamic pricing factors,
the adjusted fare before fare bounds, and the final fare after applying the
minimum and maximum fare limits.

In [139]:
final_pricing_table = flights[
    [
        "flight_id",
        "base_fare",
        "time_factor",
        "demand_factor",
        "capacity_factor",
        "seasonal_factor",
        "weekend_factor",
        "adjusted_fare",
        "final_fare"
    ]
].copy()

final_pricing_table

,flight_id,base_fare,time_factor,demand_factor,capacity_factor,seasonal_factor,weekend_factor,adjusted_fare,final_fare
0,PA101,370.52,0.95,1.0,0.9,1.00,1.0,316.79,316.79
1,PA102,433.00,0.95,0.9,1.3,1.15,1.1,608.82,600.00
2,PA103,244.08,1.08,0.9,1.3,1.15,1.0,354.68,354.68
3,PA104,169.16,1.25,0.9,0.9,1.15,1.1,216.66,216.66
4,PA105,790.30,0.95,0.9,0.9,1.15,1.0,699.36,600.00
...,...,...,...,...,...,...,...,...,...
195,PA296,784.49,0.95,1.2,1.0,1.00,1.1,983.75,600.00
196,PA297,450.73,0.95,1.2,1.0,1.00,1.0,513.83,513.83
197,PA298,314.96,1.18,0.9,0.9,1.15,1.0,346.19,346.19
198,PA299,722.75,0.95,1.2,0.9,1.00,1.1,815.70,600.00
